# 00 — PyCO2SYS foundations

**Learning goal:** use two carbonate-system inputs and fixed thermodynamic
choices to calculate and interpret DIC, TA, aqueous CO2, pH and saturation.
The Present, RCP8.5 and OAE examples below are **static chemistry calculations**.
They prescribe equilibrium input pairs; coupled OA/OAE histories, gas-exchange
rates, pump attribution and sediments are studied in 04.

See the [PyCO2SYS input-pair documentation](https://pyco2sys.readthedocs.io/en/latest/co2sys_nd/).

## Specify the conditions

Two carbonate-system inputs are sufficient only after temperature, salinity,
pressure and equilibrium-constant choices have been specified. PyCO2SYS has
defaults, but we pass these conditions explicitly so the assumptions are visible.

Use the shared template: **T = 16 °C, S = 35, P = 0 dbar**, carbonic constants
option 10, seawater pH scale 2 and buffer mode 1. The earlier examples used
15 °C; retaining the shared 16 °C setting makes this version comparable with
01/02. Set `TEMPERATURE_C = 15.0` below to revisit the earlier temperature.
PyCO2SYS pressure is in dbar; the shared configuration stores pressure in bar.
Hold the conditions fixed across the three examples to isolate chemistry.

In [10]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'teaching_config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import PyCO2SYS as pyco2
from teaching_config import TEACHING as C

# TEMPERATURE_C = C.temperature       # 16.0 degC
TEMPERATURE_C = 15.0
SALINITY = C.salinity               # 35.0, practical salinity
PRESSURE_DBAR = 10 * C.pressure_bar  # 0.0 dbar
INITIAL_TA = 2100.0                 # umol/kg; prescribed example
PRESENT_XCO2 = 430.0                # ppm; illustrative present-day input
final_co2_rcp85 = 935.0             # ppm; retained RCP8.5 example input

print(f'T = {TEMPERATURE_C:g} degC, S = {SALINITY:g}, P = {PRESSURE_DBAR:g} dbar')
print('Carbonate settings:', C.chemistry)

def show_state(result):
    for key, label, unit in (
        ('pH', 'Seawater-scale pH', ''),
        ('dic', 'DIC', 'umol/kg'),
        ('alkalinity', 'TA', 'umol/kg'),
        ('aqueous_CO2', 'Aqueous CO2', 'umol/kg'),
        ('xCO2', 'Dry-air xCO2', 'ppm'),
        ('saturation_calcite', 'Calcite saturation state', ''),
        ('saturation_aragonite', 'Aragonite saturation state', ''),
        ('revelle_factor', 'Revelle factor', ''),
    ):
        print(f'{label}: {float(result[key]):.2f} {unit}'.rstrip())

T = 15 degC, S = 35, P = 0 dbar
Carbonate settings: {'opt_k_carbonic': 10, 'opt_pH_scale': 2, 'opt_buffers_mode': 1}


## 1. Present

Prescribe TA = 2100 µmol/kg and dry-air xCO2 = 430 ppm, as in the original
example. These are illustrative inputs, not a measurement of a specific
water parcel or a live atmospheric observation. DIC, aqueous CO2, pH and
calcite/aragonite saturation are calculated outputs.

`par1_type=1` denotes TA; `par2_type=9` denotes dry-air xCO2 in ppm.
This differs from `par2_type=4`, which denotes pCO2 in µatm.

In [11]:
present = pyco2.sys(
    par1=INITIAL_TA, par1_type=1,
    par2=PRESENT_XCO2, par2_type=9,
    temperature=TEMPERATURE_C,
    salinity=SALINITY,
    pressure=PRESSURE_DBAR,
    **C.chemistry,
)
show_state(present)
target_aragonite_saturation_state_for_OAE = float(present['saturation_aragonite'])

Seawater-scale pH: 7.98
DIC: 1918.69 umol/kg
TA: 2100.00 umol/kg
Aqueous CO2: 15.78 umol/kg
Dry-air xCO2: 430.00 ppm
Calcite saturation state: 3.07
Aragonite saturation state: 1.97
Revelle factor: 11.99


## 2. RCP8.5

Retain TA and the same T, S and P, but prescribe xCO2 = 935 ppm. This is the
earlier RCP8.5-labelled high-CO2 endpoint example, not a transient scenario
run or a new forecast. Predict how DIC, aqueous CO2, pH and saturation change.
Compare the fractional changes in DIC and xCO2: the Revelle factor describes
their local sensitivity at fixed TA, rather than an exact ratio for this
large perturbation.

In [12]:
rcp85 = pyco2.sys(
    par1=INITIAL_TA, par1_type=1,
    par2=final_co2_rcp85, par2_type=9,
    temperature=TEMPERATURE_C,
    salinity=SALINITY,
    pressure=PRESSURE_DBAR,
    **C.chemistry,
)
show_state(rcp85)
print(f"pH change from Present: {float(rcp85['pH'] - present['pH']):.2f}")

Seawater-scale pH: 7.68
DIC: 2027.78 umol/kg
TA: 2100.00 umol/kg
Aqueous CO2: 34.32 umol/kg
Dry-air xCO2: 935.00 ppm
Calcite saturation state: 1.66
Aragonite saturation state: 1.07
Revelle factor: 16.30
pH change from Present: -0.30


## 3. OAE: recover the original aragonite saturation

Now use a different input pair: the Present aragonite saturation state
(`par1_type=11`) and the high-CO2 value of 935 ppm (`par2_type=9`). Infer the
TA required at those conditions and subtract the original 2100 µmol/kg.

Aragonite saturation is now a **prescribed target**, while TA and DIC are
calculated. Matching that target is a chemistry consistency check, not an
independent prediction of OAE success. Because atmospheric xCO2 remains
prescribed, DIC can change between equilibrium states; this is not a closed
carbon-inventory calculation or a simulation of an alkalinity-addition rate.
Does restoring aragonite saturation also restore the original pH and DIC?

In [13]:
oae = pyco2.sys(
    par1=target_aragonite_saturation_state_for_OAE, par1_type=11,
    par2=final_co2_rcp85, par2_type=9,
    temperature=TEMPERATURE_C,
    salinity=SALINITY,
    pressure=PRESSURE_DBAR,
    **C.chemistry,
)
show_state(oae)
required_ta_enhancement = float(oae['alkalinity']) - INITIAL_TA
print(f'Required enhancement in total alkalinity: {required_ta_enhancement:.2f} umol/kg')
print(f"pH change from Present: {float(oae['pH'] - present['pH']):.2f}")
assert abs(float(oae['saturation_aragonite'])
           - target_aragonite_saturation_state_for_OAE) < 1e-8

Seawater-scale pH: 7.81
DIC: 2779.11 umol/kg
TA: 2922.35 umol/kg
Aqueous CO2: 34.32 umol/kg
Dry-air xCO2: 935.00 ppm
Calcite saturation state: 3.07
Aragonite saturation state: 1.97
Revelle factor: 15.61
Required enhancement in total alkalinity: 822.35 umol/kg
pH change from Present: -0.17
